In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "aerolineas_dev")
dbutils.widgets.text("source", "bronze")
dbutils.widgets.text("sink", "silver")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

In [0]:
# Tablas de hechos
df_bbdd_fact_vuelos = spark.table(f"{catalog}.{schema_source}.bbdd_fact_vuelos")
df_bbdd_fact_pasajeros = spark.table(f"{catalog}.{schema_source}.bbdd_fact_pasajeros")
df_bbdd_fact_combustible = spark.table(f"{catalog}.{schema_source}.bbdd_fact_combustible")
df_bbdd_fact_ventas_boletos = spark.table(f"{catalog}.{schema_source}.bbdd_fact_ventas_boletos")
df_bbdd_fact_equipajes = spark.table(f"{catalog}.{schema_source}.bbdd_fact_equipajes")

In [0]:
# Hechos Silver
target_table_fact_vuelos = f"{catalog}.{schema_sink}.tbl_fact_vuelos"
target_table_fact_pasajeros = f"{catalog}.{schema_sink}.tbl_fact_pasajeros"
target_table_fact_combustible = f"{catalog}.{schema_sink}.tbl_fact_combustible"
target_table_fact_ventas_boletos = f"{catalog}.{schema_sink}.tbl_fact_ventas_boletos"
target_table_fact_equipajes = f"{catalog}.{schema_sink}.tbl_fact_equipajes"

## Parametros de limpieza inicial

In [0]:

def validar_dimension(df, condicion_valida, claves, nombre):
    # Mostrar registros que incumplen las reglas
    df_invalidos = df.filter(
        ~F.coalesce(condicion_valida, F.lit(False))
    )

    if df_invalidos.limit(1).count() > 0:
        display(df_invalidos)
        raise ValueError(f"{nombre}: existen registros inválidos.")

    # Validar unicidad de las claves
    for clave in claves:
        duplicados = (
            df.groupBy(clave)
            .count()
            .filter(F.col("count") > 1)
        )

        if duplicados.limit(1).count() > 0:
            display(duplicados)
            raise ValueError(
                f"{nombre}: existen duplicados en {clave}."
            )


def validar_referencia(df, columna, df_referencia, clave, nombre):
    # Detectar claves que no existen en la dimensión de referencia
    referencias = df_referencia.select(
        F.col(clave).alias(columna)
    ).distinct()

    df_sin_referencia = df.join(
        referencias,
        on=columna,
        how="left_anti"
    )

    if df_sin_referencia.limit(1).count() > 0:
        display(df_sin_referencia)
        raise ValueError(
            f"{nombre}: existen referencias inválidas en {columna}."
        )


def validar_no_negativos(df, columnas, nombre):
    condicion = F.lit(True)

    for columna in columnas:
        condicion = condicion & (F.col(columna) >= 0)

    invalidos = df.filter(
        ~F.coalesce(condicion, F.lit(False))
    )

    if invalidos.limit(1).count() > 0:
        display(invalidos)
        raise ValueError(
            f"{nombre}: existen valores negativos o nulos en campos numéricos."
        )

### Lectura de tablas dimensionales en silver

In [0]:
df_dim_aerolineas_silver = spark.table(
    f"{catalog}.{schema_sink}.tbl_dim_aerolineas"
)

df_dim_aeropuertos_silver = spark.table(
    f"{catalog}.{schema_sink}.tbl_dim_aeropuertos"
)

df_dim_aeronaves_silver = spark.table(
    f"{catalog}.{schema_sink}.tbl_dim_aeronaves"
)

df_dim_rutas_silver = spark.table(
    f"{catalog}.{schema_sink}.tbl_dim_rutas"
)

## Tranformaciones para la tabla tbl_fact_vuelos
### Normaliza identificadores, valida estados y fechas, y comprueba las referencias a las dimensiones.

In [0]:
df_fact_vuelos_silver = df_bbdd_fact_vuelos.select(
    F.upper(F.trim("vuelo_id")).alias("vuelo_id"),
    F.upper(F.trim("aerolinea_id")).alias("aerolinea_id"),
    F.upper(F.trim("aeronave_id")).alias("aeronave_id"),
    F.upper(F.trim("ruta_id")).alias("ruta_id"),
    F.upper(F.trim("numero_vuelo")).alias("numero_vuelo"),
    "fecha_salida_programada_utc",
    "fecha_salida_real_utc",
    "fecha_llegada_programada_utc",
    "fecha_llegada_real_utc",
    F.upper(F.trim("estado_vuelo")).alias("estado_vuelo"),
    F.when(
        F.length(F.trim("motivo_cancelacion")) > 0,
        F.upper(F.trim("motivo_cancelacion"))
    ).alias("motivo_cancelacion"),
    F.upper(F.trim("terminal_salida")).alias("terminal_salida"),
    F.when(
        F.length(F.trim("puerta_embarque")) > 0,
        F.upper(F.trim("puerta_embarque"))
    ).alias("puerta_embarque"),
    "demora_salida_min",
    "demora_llegada_min",
    F.upper(F.trim("fuente_sistema")).alias("fuente_sistema"),
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

es_cancelado = F.col("estado_vuelo") == "CANCELADO"
es_operado = F.col("estado_vuelo").isin("REALIZADO", "DEMORADO")

condicion_vuelos = (
    (F.length("vuelo_id") > 0)
    & (F.length("aerolinea_id") > 0)
    & (F.length("aeronave_id") > 0)
    & (F.length("ruta_id") > 0)
    & (F.length("numero_vuelo") > 0)
    & F.col("estado_vuelo").isin("REALIZADO", "DEMORADO", "CANCELADO")
    & F.col("fecha_salida_programada_utc").isNotNull()
    & (
        F.col("fecha_llegada_programada_utc")
        > F.col("fecha_salida_programada_utc")
    )
    & (
        (
            es_cancelado
            & F.col("fecha_salida_real_utc").isNull()
            & F.col("fecha_llegada_real_utc").isNull()
            & (F.length("motivo_cancelacion") > 0)
        )
        | (
            es_operado
            & F.col("fecha_salida_real_utc").isNotNull()
            & (
                F.col("fecha_llegada_real_utc")
                > F.col("fecha_salida_real_utc")
            )
            & F.col("motivo_cancelacion").isNull()
        )
    )
    & F.col("demora_salida_min").isNotNull()
    & F.col("demora_llegada_min").isNotNull()
    & (F.length("fuente_sistema") > 0)
)

validar_dimension(
    df_fact_vuelos_silver,
    condicion_vuelos,
    ["vuelo_id"],
    "fact_vuelos"
)

validar_referencia(
    df_fact_vuelos_silver,
    "aerolinea_id",
    df_dim_aerolineas_silver,
    "aerolinea_id",
    "fact_vuelos"
)

validar_referencia(
    df_fact_vuelos_silver,
    "aeronave_id",
    df_dim_aeronaves_silver,
    "aeronave_id",
    "fact_vuelos"
)

validar_referencia(
    df_fact_vuelos_silver,
    "ruta_id",
    df_dim_rutas_silver,
    "ruta_id",
    "fact_vuelos"
)

# Comprobar que la aeronave pertenece a la aerolínea del vuelo
pares_validos = df_dim_aeronaves_silver.select(
    "aeronave_id", "aerolinea_id"
).distinct()

vuelos_aeronave_invalida = df_fact_vuelos_silver.join(
    pares_validos,
    on=["aeronave_id", "aerolinea_id"],
    how="left_anti"
)

if vuelos_aeronave_invalida.limit(1).count() > 0:
    display(vuelos_aeronave_invalida)
    raise ValueError(
        "fact_vuelos: la aeronave no pertenece a la aerolínea indicada."
    )

target_table = target_table_fact_vuelos

df_fact_vuelos_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)

## Tranformaciones para la tabla tbl_fact_pasajeros
### Valida cantidades, ocupación y coherencia entre los grupos de pasajeros.

In [0]:
df_fact_pasajeros_silver = df_bbdd_fact_pasajeros.select(
    F.upper(F.trim("vuelo_id")).alias("vuelo_id"),
    "adultos",
    "ninos",
    "infantes_sin_asiento",
    "pasajeros_con_asiento",
    "pasajeros_total_a_bordo",
    "asientos_disponibles",
    "pasajeros_business",
    "pasajeros_economy",
    "transito_conexion",
    "pasajeros_no_show",
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

validar_no_negativos(
    df_fact_pasajeros_silver,
    [
        "adultos",
        "ninos",
        "infantes_sin_asiento",
        "pasajeros_con_asiento",
        "pasajeros_total_a_bordo",
        "asientos_disponibles",
        "pasajeros_business",
        "pasajeros_economy",
        "transito_conexion",
        "pasajeros_no_show"
    ],
    "fact_pasajeros"
)

condicion_pasajeros = (
    (F.length("vuelo_id") > 0)
    & (F.col("asientos_disponibles") > 0)
    & (
        F.col("adultos") + F.col("ninos")
        == F.col("pasajeros_con_asiento")
    )
    & (
        F.col("pasajeros_con_asiento") + F.col("infantes_sin_asiento")
        == F.col("pasajeros_total_a_bordo")
    )
    & (
        F.col("pasajeros_business") + F.col("pasajeros_economy")
        == F.col("pasajeros_con_asiento")
    )
    & (
        F.col("pasajeros_con_asiento")
        <= F.col("asientos_disponibles")
    )
    & (
        F.col("transito_conexion")
        <= F.col("pasajeros_total_a_bordo")
    )
)

validar_dimension(
    df_fact_pasajeros_silver,
    condicion_pasajeros,
    ["vuelo_id"],
    "fact_pasajeros"
)

validar_referencia(
    df_fact_pasajeros_silver,
    "vuelo_id",
    df_fact_vuelos_silver,
    "vuelo_id",
    "fact_pasajeros"
)

target_table = target_table_fact_pasajeros

df_fact_pasajeros_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)

## Tranformaciones para la tabla tbl_fact_combustible
### Normaliza textos, valida valores y comprueba el balance de combustible y el costo de carga. Se permite una diferencia de 0.01 por redondeo.

In [0]:
df_fact_combustible_silver = df_bbdd_fact_combustible.select(
    F.upper(F.trim("vuelo_id")).alias("vuelo_id"),
    F.upper(F.trim("tipo_combustible")).alias("tipo_combustible"),
    "combustible_inicial_l",
    "combustible_cargado_l",
    "combustible_final_l",
    "combustible_consumido_l",
    "precio_usd_l",
    "costo_carga_usd",
    F.trim("proveedor").alias("proveedor"),
    F.upper(F.trim("moneda")).alias("moneda"),
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

validar_no_negativos(
    df_fact_combustible_silver,
    [
        "combustible_inicial_l",
        "combustible_cargado_l",
        "combustible_final_l",
        "combustible_consumido_l",
        "precio_usd_l",
        "costo_carga_usd"
    ],
    "fact_combustible"
)

condicion_combustible = (
    (F.length("vuelo_id") > 0)
    & (F.length("tipo_combustible") > 0)
    & (F.length("proveedor") > 0)
    & (F.col("moneda") == "USD")
    & (F.col("precio_usd_l") > 0)
    & (
        F.abs(
            F.col("combustible_inicial_l")
            + F.col("combustible_cargado_l")
            - F.col("combustible_consumido_l")
            - F.col("combustible_final_l")
        ) <= F.lit("0.01").cast("decimal(18,2)")
    )
    & (
        F.abs(
            F.col("costo_carga_usd")
            - F.round(
                F.col("combustible_cargado_l") * F.col("precio_usd_l"),
                2
            )
        ) <= F.lit("0.01").cast("decimal(18,2)")
    )
)

validar_dimension(
    df_fact_combustible_silver,
    condicion_combustible,
    ["vuelo_id"],
    "fact_combustible"
)

validar_referencia(
    df_fact_combustible_silver,
    "vuelo_id",
    df_fact_vuelos_silver,
    "vuelo_id",
    "fact_combustible"
)

target_table = target_table_fact_combustible

df_fact_combustible_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)

## Tranformaciones para la tabla tbl_fact_ventas_boletos
### Normaliza códigos, valida cantidades e importes y comprueba que cada venta corresponda a un vuelo existente. La clave única es venta_id, porque un vuelo puede tener varias ventas.

In [0]:
df_fact_ventas_boletos_silver = df_bbdd_fact_ventas_boletos.select(
    F.upper(F.trim("venta_id")).alias("venta_id"),
    F.upper(F.trim("vuelo_id")).alias("vuelo_id"),
    "fecha_venta_utc",
    F.upper(F.trim("clase_tarifa")).alias("clase_tarifa"),
    F.upper(F.trim("canal_venta")).alias("canal_venta"),
    "boletos_vendidos",
    "boletos_volados",
    "boletos_no_show",
    "tarifa_promedio_usd",
    "ingreso_bruto_usd",
    "descuentos_usd",
    "impuestos_usd",
    "ingreso_neto_usd",
    F.upper(F.trim("moneda")).alias("moneda"),
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

validar_no_negativos(
    df_fact_ventas_boletos_silver,
    [
        "boletos_vendidos",
        "boletos_volados",
        "boletos_no_show",
        "tarifa_promedio_usd",
        "ingreso_bruto_usd",
        "descuentos_usd",
        "impuestos_usd",
        "ingreso_neto_usd"
    ],
    "fact_ventas_boletos"
)

condicion_ventas = (
    (F.length("venta_id") > 0)
    & (F.length("vuelo_id") > 0)
    & F.col("fecha_venta_utc").isNotNull()
    & F.col("clase_tarifa").isin("BUSINESS", "ECONOMY")
    & (F.length("canal_venta") > 0)
    & (F.col("moneda") == "USD")
    & (
        F.col("boletos_volados") + F.col("boletos_no_show")
        <= F.col("boletos_vendidos")
    )
    & (F.col("descuentos_usd") <= F.col("ingreso_bruto_usd"))
)

validar_dimension(
    df_fact_ventas_boletos_silver,
    condicion_ventas,
    ["venta_id"],
    "fact_ventas_boletos"
)

validar_referencia(
    df_fact_ventas_boletos_silver,
    "vuelo_id",
    df_fact_vuelos_silver,
    "vuelo_id",
    "fact_ventas_boletos"
)

target_table = target_table_fact_ventas_boletos

df_fact_ventas_boletos_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)

## Tranformaciones para la tabla tbl_fact_equipajes
### Valida piezas, peso e ingresos. Conserva NULL en el tiempo de entrega cuando no hubo entrega de equipaje; no lo reemplaza por cero.

In [0]:
df_fact_equipajes_silver = df_bbdd_fact_equipajes.select(
    F.upper(F.trim("vuelo_id")).alias("vuelo_id"),
    "piezas_facturadas",
    "peso_facturado_kg",
    "piezas_cabina_estimadas",
    "piezas_extraviadas",
    "piezas_danadas",
    "ingreso_equipaje_usd",
    "tiempo_entrega_promedio_min",
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

validar_no_negativos(
    df_fact_equipajes_silver,
    [
        "piezas_facturadas",
        "peso_facturado_kg",
        "piezas_cabina_estimadas",
        "piezas_extraviadas",
        "piezas_danadas",
        "ingreso_equipaje_usd"
    ],
    "fact_equipajes"
)

condicion_equipajes = (
    (F.length("vuelo_id") > 0)
    & (F.col("piezas_extraviadas") <= F.col("piezas_facturadas"))
    & (F.col("piezas_danadas") <= F.col("piezas_facturadas"))
    & (
        F.col("tiempo_entrega_promedio_min").isNull()
        | (F.col("tiempo_entrega_promedio_min") >= 0)
    )
)

validar_dimension(
    df_fact_equipajes_silver,
    condicion_equipajes,
    ["vuelo_id"],
    "fact_equipajes"
)

validar_referencia(
    df_fact_equipajes_silver,
    "vuelo_id",
    df_fact_vuelos_silver,
    "vuelo_id",
    "fact_equipajes"
)

target_table = target_table_fact_equipajes

df_fact_equipajes_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)